In [1]:
# Cell 1: Imports and Setup
import sys
import os
import random
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from collections import defaultdict
from typing import List
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../src"))

from abstractions3d.primitives.shapes import Box3D
from abstractions3d.primitives.visualization import visualize_boxes_3d
from abstractions3d.dsl.core import Shape3D
from abstractions3d.dsl.nodes import Rect3D, Move3D, Union3D, SymTrans3D, SymRef3D
from abstractions3d.dsl.instantiation import instantiate_3d
from abstractions3d.data.blueprint import Table3D, Chair3D, Bench3D

# Set random seeds
random.seed(42)
torch.manual_seed(42)

In [2]:
import random
from typing import List
from abstractions3d.data.blueprint import Table3D, Chair3D, Bench3D
from abstractions3d.dsl.nodes import Union3D

def sample_uniform(low: float, high: float) -> float:
    return random.uniform(low, high)

def generate_dataset(num_samples: int = 50) -> List[Union3D]:
    dataset = []
    for _ in range(num_samples):
        dataset.append(Table3D(
            top_length=sample_uniform(3.0, 6.0),
            top_depth=sample_uniform(1.5, 3.0),
            top_thickness=sample_uniform(0.1, 0.3),
            leg_length=sample_uniform(0.2, 0.4),
            leg_depth=sample_uniform(0.2, 0.4),
            leg_height=sample_uniform(1.0, 2.0)
        ))
        dataset.append(Chair3D(
            seat_length=sample_uniform(1.0, 2.0),
            seat_depth=sample_uniform(1.0, 2.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=sample_uniform(1.0, 2.0),
            backrest_thickness=sample_uniform(0.1, 0.3)
        ))
        backrest_height = sample_uniform(0.0, 1.5)
        dataset.append(Bench3D(
            seat_length=sample_uniform(2.0, 5.0),
            seat_depth=sample_uniform(0.5, 1.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=backrest_height,
            backrest_thickness=sample_uniform(0.0, 0.3) if backrest_height > 0 else 0.0
        ))
    return dataset

In [3]:
# Cell 3: Utilities

def get_singletons(shapes):
    singletons = defaultdict(list)
    if isinstance(shapes, list):
        for s in shapes:
            res = get_singletons(s)
            for k, v in res.items():
                singletons[k] += v
        return singletons

    cls, params = shapes.param_tuple()
    singletons[cls.__name__].append(params)

    if hasattr(shapes, 'children'):
        for child in getattr(shapes, 'children', []):
            res = get_singletons(child)
            for k, v in res.items():
                singletons[k] += v
    return singletons

def get_pairs(shapes):
    pairs = defaultdict(list)
    if isinstance(shapes, list):
        for s in shapes:
            res = get_pairs(s)
            for k, v in res.items():
                pairs[k] += v
        return pairs

    if hasattr(shapes, 'children') and len(shapes.children) == 2:
        cls, (child1, child2) = shapes.param_tuple()
        type_str = f"{cls.__name__}({child1.param_tuple()[0].__name__},{child2.param_tuple()[0].__name__})"
        pairs[type_str].append(child1.param_tuple()[1] + child2.param_tuple()[1])
        for child in shapes.children:
            res = get_pairs(child)
            for k, v in res.items():
                pairs[k] += v
    return pairs

def prepare_autoencoder_train_data(parameters, mask):
    tensor = torch.tensor(parameters, dtype=torch.float32)
    masked_data = tensor[mask]
    if len(masked_data) == 0:
        return None
    dataset = TensorDataset(masked_data)
    return DataLoader(dataset, batch_size=64, shuffle=True)

def is_well_explained(parameters, model, threshold=0.01):
    with torch.no_grad():
        recon = model(parameters)
        error = torch.max(torch.abs(recon - parameters), dim=1).values
        return error < threshold

In [4]:
class Abstraction(Shape3D):
    def __init__(self, type_list, float_parameters, other_parameters, model):
        self.type_list = type_list
        self.float_parameters = float_parameters
        self.other_parameters = other_parameters
        self.model = model
        self.children = []  # empty to reduce node count

        # Expand immediately to store geometry for visualization
        try:
            expanded_shape = self.expand()
            self.boxes = expanded_shape.get_box3d_list()
        except Exception:
            self.boxes = []

    def __str__(self):
        s = "Abstraction(\n"
        for f in self.float_parameters:
            s += f"  {f}\n"
        for o in self.other_parameters:
            s += f"  {o}\n"
        s += ")"
        return s

    def expand(self):
        self.model.eval()
        float_tensor = torch.tensor(self.float_parameters, dtype=torch.float32).unsqueeze(0)
        decoder_output = self.model(float_tensor)
        expanded_float_parameters = decoder_output.squeeze(0).tolist()

        full_parameter_list = []
        i_floats = 0
        i_others = 0
        for t in self.type_list:
            if t == float:
                full_parameter_list.append(expanded_float_parameters[i_floats])
                i_floats += 1
            else:
                full_parameter_list.append(self.other_parameters[i_others])
                i_others += 1

        return instantiate_3d(self.type_list, full_parameter_list)

    def get_box3d_list(self):
        # Return the precomputed boxes
        return self.boxes

In [5]:
# Cell 5: Train Autoencoders

def train_abstractions(structures, retrain_iterations=2, error_threshold=0.01):
    losses = defaultdict(list)
    models = {}

    for structure_name, parameters in structures.items():
        float_indices = [i for i, t in enumerate(parameters[0]) if isinstance(t, float)]
        if not float_indices:
            continue

        float_params = [[p[i] for i in float_indices] for p in parameters]
        float_tensor = torch.tensor(float_params, dtype=torch.float32)
        well_explained = torch.ones(len(float_tensor), dtype=torch.bool)

        for _ in range(retrain_iterations):
            train_loader = prepare_autoencoder_train_data(float_tensor, well_explained)
            if train_loader is None:
                print(f"Skipping {structure_name}: no well-explained samples")
                break

            model = nn.Sequential(
                nn.Linear(len(float_indices), 32),
                nn.ReLU(),
                nn.Linear(32, 32),
                nn.ReLU(),
                nn.Linear(32, len(float_indices))
            )
            optimizer = AdamW(model.parameters(), lr=0.001, weight_decay=0.05)
            loss_fn = lambda recon, x: torch.mean(torch.max(torch.abs(recon - x), dim=1).values)

            model.train()
            for epoch in range(100):
                running_loss = 0.0
                for batch in train_loader:
                    x = batch[0]
                    optimizer.zero_grad()
                    recon = model(x)
                    loss = loss_fn(recon, x)
                    loss.backward()
                    optimizer.step()
                    running_loss += loss.item()
                losses[structure_name].append(running_loss / len(train_loader))

            model.eval()
            well_explained = is_well_explained(float_tensor, model, threshold=error_threshold)
            if torch.sum(well_explained) == 0:
                well_explained[0] = True

        models[structure_name] = model
    return models, losses

In [6]:
# Cell 6: integrate_abstractions (new version)

def integrate_abstractions(shape_or_list, models, error_threshold=0.01):
    if isinstance(shape_or_list, list):
        return [integrate_abstractions(s, models, error_threshold) for s in shape_or_list]

    if shape_or_list is None or not isinstance(shape_or_list, Shape3D):
        return shape_or_list

    shape = shape_or_list
    try:
        cls, parameters = shape.param_tuple()
    except Exception:
        return shape

    type_str = cls.__name__
    float_params = [p for p in parameters if isinstance(p, float)]
    other_params = [p for p in parameters if not isinstance(p, float)]

    fits_well = False
    if float_params and type_str in models:
        float_tensor = torch.tensor([float_params], dtype=torch.float32)
        with torch.no_grad():
            recon = models[type_str](float_tensor)
            error = torch.max(torch.abs(recon - float_tensor))
            fits_well = error < error_threshold

    if fits_well:
        type_list = [cls] + [type(p) for p in parameters]
        new_shape = Abstraction(type_list, float_params, other_params, models[type_str])
        new_shape.children = []  # remove children to reduce nodes
        return new_shape
    else:
        new_shape = cls(*parameters)
        if hasattr(new_shape, 'children') and isinstance(new_shape.children, list):
            new_shape.children = [integrate_abstractions(c, models, error_threshold)
                                  for c in new_shape.children]
        return new_shape

In [7]:
# Cell 7: Full Pipeline

def abstraction_pipeline(dataset, error_threshold=0.01):
    singletons = get_singletons(dataset)
    pairs = get_pairs(dataset)

    singleton_models, _ = train_abstractions(singletons, error_threshold=error_threshold)
    pair_models, _ = train_abstractions(pairs, error_threshold=error_threshold)

    dataset_singleton_abstracted = integrate_abstractions(dataset, singleton_models, error_threshold)
    dataset_fully_abstracted = integrate_abstractions(dataset_singleton_abstracted, pair_models, error_threshold)

    return dataset_fully_abstracted

In [8]:
# Cell 8: Run Pipeline and Visualize

dataset = generate_dataset(50)
print(f"Generated {len(dataset)} shapes.")

abstracted_dataset = abstraction_pipeline(dataset)
print("Abstraction pipeline completed.")

# Count nodes before and after
def count_nodes(shape):
    if isinstance(shape, list):
        return sum(count_nodes(s) for s in shape)
    return 1 + sum(count_nodes(c) for c in getattr(shape, 'children', []))

total_before = count_nodes(dataset)
total_after = count_nodes(abstracted_dataset)
print("Total nodes before abstraction:", total_before)
print("Total nodes after abstraction:", total_after)

# # Visualize first 5 shapes
# for shape in abstracted_dataset[:5]:
#     boxes = shape.get_box3d_list()
#     visualize_boxes_3d(boxes)

Generated 150 shapes.


/var/folders/zb/9bmh3s9s3q5_mlv24by4t5x00000gn/T/ipykernel_16648/2854117456.py:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tensor = torch.tensor(parameters, dtype=torch.float32)
/var/folders/zb/9bmh3s9s3q5_mlv24by4t5x00000gn/T/ipykernel_16648/2854117456.py:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tensor = torch.tensor(parameters, dtype=torch.float32)


Abstraction pipeline completed.
Total nodes before abstraction: 1350
Total nodes after abstraction: 1120


In [9]:
dataset[0]

In [10]:
print(dataset[0])

Union3D(
    Move3D(
        Rect3D(
            4.918,
            0.155,
            1.538
        ),
        0.000,
        1.754,
        0.000
    ),
    SymRef3D(
        SymRef3D(
            Move3D(
                Rect3D(
                    0.245,
                    1.677,
                    0.347
                ),
                2.287,
                0.838,
                0.545
            ),
            x
        ),
        z
    )
)


In [11]:
print(abstracted_dataset[0])

Union3D(
    Abstraction(
      0.0
      1.7542024192598231
      0.0
      Rect3D(
        4.918,
        0.155,
        1.538
    )
    ),
    SymRef3D(
        SymRef3D(
            Move3D(
                Rect3D(
                    0.245,
                    1.677,
                    0.347
                ),
                2.287,
                0.838,
                0.545
            ),
            x
        ),
        z
    )
)


In [16]:
visualize_boxes_3d(abstracted_dataset[0].get_box3d_list())

Output()

In [17]:
visualize_boxes_3d(dataset[0].get_box3d_list())

Output()

In [18]:
visualize_boxes_3d(abstracted_dataset[95].get_box3d_list())

Output()

In [19]:
print(abstracted_dataset[95])

Union3D(
    Abstraction(
      0.0
      1.0949999366316168
      0.0
      Rect3D(
        4.801,
        0.363,
        0.713
    )
    ),
    Union3D(
        SymRef3D(
            SymRef3D(
                Move3D(
                    Abstraction(
                      0.22952694468049614
                      0.9133155798250299
                      0.28168228904252124
                    ),
                    2.236,
                    0.457,
                    0.165
                ),
                x
            ),
            z
        ),
        Abstraction(
          0.0
          1.7725715755782292
          -0.3455732659976808
          Rect3D(
            4.801,
            0.992,
            0.021
        )
        )
    )
)


In [10]:
from collections import defaultdict

def add_dicts(d1: dict, d2: dict) -> dict:
    """Merge two dicts of lists."""
    out = defaultdict(list, d1)
    for k, v in d2.items():
        out[k].extend(v)
    return dict(out)

In [12]:
# Generate dataset
dataset = generate_dataset(50)
print(f"Generated {len(dataset)} shapes.")

# Extract structure types
singletons = get_singletons(dataset)
pairs = get_pairs(dataset)
structures = add_dicts(singletons, pairs)
print("Extracted structures:", structures.keys())

# --- Plot float parameter distributions ---
for (name, dim), params in structures.items():
    if not params: 
        continue
    params = torch.tensor(params, dtype=torch.float32)
    fig, axes = plt.subplots(1, min(dim, 4), figsize=(4*min(dim,4), 3))
    if dim == 1:
        axes = [axes]  # ensure iterable
    for i in range(min(dim,4)):
        axes[i].hist(params[:, i].numpy(), bins=20, alpha=0.7)
        axes[i].set_title(f"{name} param {i}")
    plt.suptitle(f"Distribution for {name} ({dim} floats)")
    plt.show()


Generated 150 shapes.
Extracted structures: dict_keys(['Union3D', 'Move3D', 'Rect3D', 'SymRef3D', 'Union3D(Move3D,SymRef3D)', 'Union3D(Move3D,Union3D)', 'Union3D(SymRef3D,Move3D)'])


ValueError: too many values to unpack (expected 2)

In [13]:
for name, params in structures.items():
    print(f"\n{name} ({len(params)} samples):")
    for p in params[:5]:  # show first 5 examples
        print("  ", p)


Union3D (250 samples):
   (<abstractions3d.dsl.nodes.Move3D object at 0x133824fb0>, <abstractions3d.dsl.nodes.SymRef3D object at 0x133827ce0>)
   (<abstractions3d.dsl.nodes.Move3D object at 0x1338271d0>, <abstractions3d.dsl.nodes.Union3D object at 0x133826cf0>)
   (<abstractions3d.dsl.nodes.SymRef3D object at 0x133826d50>, <abstractions3d.dsl.nodes.Move3D object at 0x1338262a0>)
   (<abstractions3d.dsl.nodes.Move3D object at 0x1338260c0>, <abstractions3d.dsl.nodes.Union3D object at 0x13384a7b0>)
   (<abstractions3d.dsl.nodes.SymRef3D object at 0x13384b800>, <abstractions3d.dsl.nodes.Move3D object at 0x13384a030>)

Move3D (400 samples):
   (<abstractions3d.dsl.nodes.Rect3D object at 0x133826c00>, 0.0, 1.5484906177367517, 0.0)
   (<abstractions3d.dsl.nodes.Rect3D object at 0x133827980>, 1.4991075299623404, 0.7287857165307706, 0.6595603496457448)
   (<abstractions3d.dsl.nodes.Rect3D object at 0x1338254c0>, 0.0, 0.8052533098968025, 0.0)
   (<abstractions3d.dsl.nodes.Rect3D object at 0x133

In [14]:
def get_singletons(shape):
    """Extract only float/int parameters from individual nodes."""
    singletons = defaultdict(list)
    if isinstance(shape, list):
        for s in shape:
            singletons = add_dicts(singletons, get_singletons(s))
        return singletons

    t, params = shape.param_tuple()
    floats = tuple(p for p in params if isinstance(p, (float, int)))
    if floats:
        singletons[t.__name__].append(floats)

    # recurse
    for c in getattr(shape, 'children', []):
        singletons = add_dicts(singletons, get_singletons(c))
    return singletons


def get_pairs(shape):
    """Extract float/int parameters from Union3D pairs only."""
    pairs = defaultdict(list)
    if isinstance(shape, list):
        for s in shape:
            pairs = add_dicts(pairs, get_pairs(s))
        return pairs

    if isinstance(shape, Union3D):
        t, (c1, c2) = shape.param_tuple()
        t1, p1 = c1.param_tuple()
        t2, p2 = c2.param_tuple()
        floats1 = [p for p in p1 if isinstance(p, (float, int))]
        floats2 = [p for p in p2 if isinstance(p, (float, int))]
        if floats1 or floats2:
            key = f"{t.__name__}({t1.__name__},{t2.__name__})"
            pairs[key].append(tuple(floats1 + floats2))
        # recurse
        for c in shape.children:
            pairs = add_dicts(pairs, get_pairs(c))
    else:
        for c in getattr(shape, 'children', []):
            pairs = add_dicts(pairs, get_pairs(c))
    return pairs

In [15]:
singletons = get_singletons(dataset)
pairs = get_pairs(dataset)
structures = add_dicts(singletons, pairs)

for k,v in structures.items():
    print(f"{k} ({len(v)}): {v[:3]}")

Move3D (400): [(0.0, 1.5484906177367517, 0.0), (1.4991075299623404, 0.7287857165307706, 0.6595603496457448), (0.0, 0.8052533098968025, 0.0)]
Rect3D (400): [(3.412095777519372, 0.1818383693504208, 1.8004448861927154), (0.31388071759469083, 1.4575714330615412, 0.38132418690122555), (1.3163834576487088, 0.43368235323014437, 1.715668456110036)]
Union3D(Move3D,SymRef3D) (50): [(0.0, 1.5484906177367517, 0.0), (0.0, 1.3459470352411202, 0.0), (0.0, 2.1051094016894787, 0.0)]
Union3D(Move3D,Union3D) (100): [(0.0, 0.8052533098968025, 0.0), (0.0, 0.758693232301386, 0.0), (0.0, 0.9867060409157653, 0.0)]
Union3D(SymRef3D,Move3D) (100): [(0.0, 1.8393439710882102, -0.8073628779126186), (0.0, 1.1785116893743743, -0.19997947833696766), (0.0, 1.948381475328787, -0.43215624707472794)]


In [16]:
import k3d
import numpy as np

def plot_structure_params_3d(structures, n_samples=200):
    """
    Interactive 3D scatter plots for float parameters of structures.
    Uses k3d for visualization.
    
    Args:
        structures (dict): {name: list of float-tuples}
        n_samples (int): max number of samples to plot per structure
    """
    plots = {}
    for name, params in structures.items():
        if not params:
            continue
        data = np.array(params[:n_samples], dtype=np.float32)
        
        if data.shape[1] == 1:  # 1D → plot along X
            x = data[:,0]
            y = np.zeros_like(x)
            z = np.zeros_like(x)
        elif data.shape[1] == 2:  # 2D → XY plane
            x, y = data[:,0], data[:,1]
            z = np.zeros_like(x)
        else:  # 3D+
            x, y, z = data[:,0], data[:,1], data[:,2]
        
        plt = k3d.plot()
        plt += k3d.points(np.vstack([x,y,z]).T, point_size=0.1, color=0x1f77b4)
        plt.display()
        plots[name] = plt

    return plots

In [18]:
# Plot all structures
plots = plot_structure_params_3d(structures, n_samples=100)

Output()

Output()

Output()

Output()

Output()

In [20]:
plots['Rect3D']

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

In [21]:
plots['Move3D']

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

In [22]:
plots['Union3D(Move3D,SymRef3D)']

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

In [23]:
plots['Union3D(Move3D,Union3D)']

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

In [24]:
plots['Union3D(SymRef3D,Move3D)']

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

In [25]:
rect_params = np.array(structures['Rect3D'], dtype=np.float32)  # shape (N,3)

In [26]:
from sklearn.cluster import KMeans

k = 3  # number of clusters
kmeans = KMeans(n_clusters=k, random_state=42)
labels = kmeans.fit_predict(rect_params)

In [27]:
import k3d

colors = np.array([
    0x1f77b4, 0xff7f0e, 0x2ca02c  # blue, orange, green
], dtype=np.uint32)

point_colors = colors[labels]

plot = k3d.plot()
plot += k3d.points(rect_params, point_size=0.1, color=point_colors)
plot.display()

TraitError: The 'color' trait of a Points instance expected an int or a dict, not the ndarray array([ 2062260, 16744206,  2924588, 16744206, 16744206,  2062260,
       16744206,  2062260,  2062260, 16744206,  2924588, 16744206,
       16744206,  2924588, 16744206,  2924588,  2062260, 16744206,
        2924588, 16744206, 16744206,  2924588, 16744206,  2924588,
        2062260, 16744206,  2924588, 16744206,  2924588,  2062260,
       16744206,  2062260,  2062260, 16744206,  2924588, 16744206,
       16744206,  2924588, 16744206,  2924588,  2062260, 16744206,
        2924588, 16744206, 16744206,  2062260, 16744206,  2062260,
        2062260, 16744206,  2924588, 16744206, 16744206,  2062260,
       16744206,  2062260,  2062260, 16744206,  2924588, 16744206,
       16744206,  2924588, 16744206,  2924588,  2062260, 16744206,
        2924588, 16744206, 16744206,  2924588, 16744206,  2924588,
        2062260, 16744206,  2924588, 16744206, 16744206,  2062260,
       16744206,  2062260,  2062260

In [28]:
point_colors = [int(c) for c in colors[labels]]  # convert ndarray → list of int

In [29]:
plot = k3d.plot()
plot += k3d.points(rect_params, point_size=0.1, color=point_colors)
plot.display()

TraitError: The 'color' trait of a Points instance expected an int or a dict, not the list [2062260, 16744206, 2924588, 16744206, 16744206, 2062260, 16744206, 2062260, 2062260, 16744206, 2924588, 16744206, 16744206, 2924588, 16744206, 2924588, 2062260, 16744206, 2924588, 16744206, 16744206, 2924588, 16744206, 2924588, 2062260, 16744206, 2924588, 16744206, 2924588, 2062260, 16744206, 2062260, 2062260, 16744206, 2924588, 16744206, 16744206, 2924588, 16744206, 2924588, 2062260, 16744206, 2924588, 16744206, 16744206, 2062260, 16744206, 2062260, 2062260, 16744206, 2924588, 16744206, 16744206, 2062260, 16744206, 2062260, 2062260, 16744206, 2924588, 16744206, 16744206, 2924588, 16744206, 2924588, 2062260, 16744206, 2924588, 16744206, 16744206, 2924588, 16744206, 2924588, 2062260, 16744206, 2924588, 16744206, 16744206, 2062260, 16744206, 2062260, 2062260, 16744206, 2924588, 16744206, 16744206, 2924588, 16744206, 2924588, 2062260, 16744206, 2924588, 16744206, 2924588, 2924588, 16744206, 2924588

In [30]:
import k3d
import numpy as np

# Example: rect_params is N x 3, labels is N-length integer array
points = np.array(rect_params, dtype=np.float32)
labels = np.array(labels, dtype=np.int32)  # cluster labels

plot = k3d.plot()

# k3d supports color via 'attribute', which is a scalar per point
plot += k3d.points(points, point_size=0.1, attribute=labels, color_map=k3d.basic_color_maps.Jet)
plot.display()

Output()

In [31]:
from sklearn.cluster import DBSCAN
import numpy as np

In [32]:
rect_params = np.array(structures['Rect3D'], dtype=np.float32)  # N x 3

In [33]:
db = DBSCAN(eps=0.5, min_samples=5)  # tune eps/min_samples
labels = db.fit_predict(rect_params)
num_clusters = len(set(labels)) - (1 if -1 in labels else 0)
print(f"Detected clusters: {num_clusters}, including noise")

Detected clusters: 4, including noise


In [34]:
import k3d

plot = k3d.plot()
plot.display()

# assign colors (use Jet colormap)
plot += k3d.points(rect_params, point_size=0.1, attribute=labels, color_map=k3d.basic_color_maps.Jet)

Output()